# First Language Model With OLM

<a href="https://colab.research.google.com/github/openlanguagemodel/openlanguagemodel/blob/main/notebooks/01_first_language_model_colab.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

This notebook trains a tiny causal language model on a local text file,
samples from it, saves it, and loads it back.

It is intentionally small. The point is to see the whole loop:
data -> tokenizer -> model -> trainer -> generation -> save/load.

## Install OLM

In Colab, this installs the latest GitHub version. If you are running
inside a local checkout with OLM already installed, this cell does
nothing.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("olm") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/openlanguagemodel/openlanguagemodel.git",
    ])

## Imports And Reproducibility

In [ ]:
import math
import os
import random
import shutil
from pathlib import Path

import torch

from olm.data.datasets import DataLoader, LocalTextDataset
from olm.data.tokenization import HFTokenizer
from olm.nn.blocks import LM
from olm.nn.structure import load_model
from olm.train import Trainer
from olm.train.optim import AdamW

seed = 42
random.seed(seed)
torch.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Make A Tiny Local Dataset

`LocalTextDataset` reads `.txt` files from a directory and turns the
stream into `(input_ids, labels)` pairs for next-token prediction.

In [ ]:
data_dir = Path("tiny_olm_data")
data_dir.mkdir(exist_ok=True)

seed_text = '''
Language models learn by predicting the next token.
A transformer reads a sequence, mixes information with attention,
and writes new hidden states through feed-forward layers.
OLM keeps these pieces visible. Embeddings, attention, norms,
residual paths, output heads, and training loops are all ordinary
PyTorch modules.
'''

repeated = "\n".join(seed_text.strip() for _ in range(500))
(data_dir / "tiny.txt").write_text(repeated, encoding="utf-8")

print((data_dir / "tiny.txt").read_text(encoding="utf-8")[:500])

## Tokenizer, Dataset, And Loader

In [ ]:
tokenizer = HFTokenizer("gpt2")
context_length = 128

dataset = LocalTextDataset(
    data_dir,
    tokenizer,
    context_length=context_length,
    shuffle=True,
    seed=seed,
)

loader = DataLoader(
    dataset,
    batch_size=8,
    num_workers=0,
    pin_memory=device.startswith("cuda"),
)

x, y = next(iter(loader))
print("input batch:", tuple(x.shape), x.dtype)
print("label batch:", tuple(y.shape), y.dtype)
print("decoded example:")
print(tokenizer.decode(x[0]))

## Build A Small LM

`LM` is a compact GPT-style model assembled from OLM blocks. Its
output head ties weights to the input token embedding by default.

In [ ]:
model = LM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=context_length,
    dropout=0.0,
)

params = sum(p.numel() for p in model.parameters())
print(f"parameters: {params:,}")

# The output projection reuses the token embedding matrix.
print("tied output head:", model.blocks[-1].weight is model.blocks[0].embedding.weight)

## A Tiny Generation Helper

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=80, temperature=0.8, top_k=50):
    model.eval()
    input_ids = tokenizer.encode(prompt).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        idx = input_ids[:, -context_length:]
        logits = model(idx)[:, -1, :]
        logits = logits / max(temperature, 1e-6)

        if top_k is not None:
            k = min(top_k, logits.size(-1))
            values, _ = torch.topk(logits, k)
            logits[logits < values[:, [-1]]] = -float("inf")

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

    return tokenizer.decode(input_ids[0].cpu())

model = model.to(device)
print(generate(model, tokenizer, "Language models", max_new_tokens=40))

## Train

This is deliberately short. Increase `max_steps` if you want the
generated text to become less chaotic.

In [ ]:
trainer = Trainer(
    model,
    AdamW,
    loader,
    device=device,
    context_length=context_length,
    learning_rate=3e-4,
    weight_decay=0.1,
    grad_accum_steps=1,
    use_amp=device.startswith("cuda"),
    grad_clip_norm=1.0,
    use_warmup_cosine=True,
)

losses = trainer.train(epochs=1, max_steps=50, log_interval=10)
print("first loss:", losses[0])
print("last loss:", losses[-1])

## Generate After Training

In [ ]:
print(generate(model, tokenizer, "Language models", max_new_tokens=100))

## Save And Load

In [ ]:
save_dir = Path("tiny_olm_model")
if save_dir.exists():
    shutil.rmtree(save_dir)

model.cpu().save(str(save_dir), tokenizer=tokenizer)
loaded_model, loaded_tokenizer = load_model(str(save_dir))
loaded_model = loaded_model.to(device)

print("saved files:", sorted(p.name for p in save_dir.iterdir()))
print(generate(loaded_model, loaded_tokenizer, "A transformer", max_new_tokens=80))

## What To Try Next

- Change `embed_dim`, `num_layers`, or `num_heads`.
- Increase `max_steps`.
- Replace the local text file with your own notes or essays.
- Move to the FineWeb-Edu notebook for a real pretraining dataset.